# FX40 spectroscopy loader

Loads TIF files from the NX40 using their provided code and converts them to a dataframe

Make sure TIFF_Read_Write.py is in the same folder

TIFF_Read_Write.py found [here](https://spmdevhub.com/snippets/reading-and-writing-tiff-files/) on 23-09-2026

In [1]:
import os
import numpy as np
from pspylib.tiff.reader import TiffReader
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from TIFF_Read_Write import Read_tiff

g_0 = 7.748e-5

## Load with no processing

In [8]:
dataFolder = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-11-05 new 100ugml 1hr\AFM\CAFM\-2.5V - 3V\Calibration"

if dataFolder[-1] != "/":
    dataFolder = dataFolder + "/"

print(dataFolder)

results = []

files = sorted(
    f for f in os.listdir(dataFolder)
    if os.path.isfile(os.path.join(dataFolder, f)) and f.lower().endswith('.tiff')
)

reader = Read_tiff()

for file in tqdm(files):
    spectDict, spectTiff = reader(os.path.join(dataFolder,file))
    v = spectDict['SPECT_DATA'][0][0]
    j = spectDict['SPECT_DATA'][0][1]
    g = np.gradient(j*1e-9,v)
    logg = np.log10(np.abs(g/g_0))
    logj = np.log10(np.abs(j*1e-9))

    results.append({
        "filename": file,
        "Bias": v.tolist(),
        "Current": (j).tolist(), 
        "dIdV": g.tolist(),
        "Log(G/G0)": logg.tolist(),
        "Log(Current)": logj.tolist()
    })

processedFolder = dataFolder + r'\Processed'
if not os.path.exists(processedFolder):
    os.makedirs(processedFolder)
df_all = pd.DataFrame(results)
print(df_all.head())
df_all.to_pickle(dataFolder + r'\Processed\FX40_Spectroscopy_Data.pkl')

print(f"Loaded {len(os.listdir(dataFolder))} ramp files")


C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-11-05 new 100ugml 1hr\AFM\CAFM\-2.5V - 3V\Calibration/


  0%|          | 0/492 [00:00<?, ?it/s]

C:\Users\newsonj\AppData\Local\Temp\ipykernel_29872\35305397.py:22: RuntimeWarning: divide by zero encountered in log10
  logg = np.log10(np.abs(g/g_0))


               filename                                               Bias  \
0  IV_261106_3311_.tiff  [0.002640010789036751, 0.011000181548297405, 0...   
1  IV_261106_3312_.tiff  [0.002640010789036751, 0.011000181548297405, 0...   
2  IV_261106_3313_.tiff  [0.002640010789036751, 0.011000181548297405, 0...   
3  IV_261106_3314_.tiff  [0.002640010789036751, 0.011000181548297405, 0...   
4  IV_261106_3315_.tiff  [0.002640010789036751, 0.011000181548297405, 0...   

                                             Current  \
0  [0.015903672203421593, 0.0479411706328392, 0.0...   
1  [0.0172820296138525, 0.009784882888197899, 0.0...   
2  [0.0053517878986895084, 0.01278374157845974, -...   
3  [0.0009186923271045089, 0.00822957418859005, 0...   
4  [0.004671922419220209, 0.023112108930945396, 0...   

                                                dIdV  \
0  [3.83215837953182e-09, 5.02228045412925e-10, -...   
1  [-8.967695686538376e-10, 5.477158703624201e-10...   
2  [8.889715167047007e-10,

## load and filter data

In [ ]:
dataFolder = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\2026-10-02 JN fP2\CAFM\-2V - 2V\Area 3"

if dataFolder[-1] != "/":
    dataFolder = dataFolder + "/"

print(dataFolder)

SCBiasRange = [0,1] # define the bias range for filtering short circuits
OCBiasRange = [1.8,2] # define the bias range for filtering open circuits

maxCurrent = 9900 # max current in nA for filtering short circuits
minCurrent = 0.5 # min current in nA for filtering open circuits

results = []

files = sorted(
    f for f in os.listdir(dataFolder)
    if os.path.isfile(os.path.join(dataFolder, f)) and f.lower().endswith('.tiff')
)

reader = Read_tiff()

for file in tqdm(files):
    spectDict, spectTiff = reader(os.path.join(dataFolder,file))
    v = spectDict['SPECT_DATA'][0][0]
    j = spectDict['SPECT_DATA'][0][1]

    if any((SCBiasRange[0] < np.abs(v[i]) < SCBiasRange[1] and np.abs(j[i]) > maxCurrent) for i in range(len(v))):
        continue  # Skip this file if it meets the short circuit condition

    if any((OCBiasRange[0] < np.abs(v[i]) < OCBiasRange[1] and np.abs(j[i]) < minCurrent) for i in range(len(v))):
        continue  # Skip this file if it meets the open circuit condition

    g = np.gradient(j*1e-9,v)
    logg = np.log10(np.abs(g/g_0))
    logj = np.log10(np.abs(j*1e-9))

    results.append({
        "filename": file,
        "Bias": v.tolist(),
        "Current": (j).tolist(), 
        "dIdV": g.tolist(),
        "Log(G/G0)": logg.tolist(),
        "Log(Current)": logj.tolist()
    })

processedFolder = dataFolder + r'\Processed'
if not os.path.exists(processedFolder):
    os.makedirs(processedFolder)
df_all = pd.DataFrame(results)
print(df_all.head())
df_all.to_pickle(dataFolder + r'\Processed\FX40_Spectroscopy_Data_Filtered.pkl')

print(f"Loaded {len(os.listdir(dataFolder))} ramp files")


## Load and subtract a reference average

In [20]:
calibrationFile = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\AFM\CAFM\-2V - 2V\calibration\Processed\FX40_Spectroscopy_Data.pkl"
calib = pd.read_pickle(calibrationFile)
currentAvg = np.nanmean(np.array(calib["Current"].tolist()),axis=0)

dataFolder = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\AFM\CAFM\-2V - 2V\Area 16"

if dataFolder[-1] != "/":
    dataFolder = dataFolder + "/"

print(dataFolder)

results = []

files = sorted(
    f for f in os.listdir(dataFolder)
    if os.path.isfile(os.path.join(dataFolder, f)) and f.lower().endswith('.tiff')
)

reader = Read_tiff()

for file in tqdm(files):
    spectDict, spectTiff = reader(os.path.join(dataFolder,file))
    v = spectDict['SPECT_DATA'][0][0]
    j = spectDict['SPECT_DATA'][0][1]-currentAvg
    g = np.gradient(j*1e-9,v)
    logg = np.log10(np.abs(g/g_0))
    logj = np.log10(np.abs(j*1e-9))

    results.append({
        "filename": file,
        "Bias": v.tolist(),
        "Current": (j).tolist(), 
        "dIdV": g.tolist(),
        "Log(G/G0)": logg.tolist(),
        "Log(Current)": logj.tolist()
    })

processedFolder = dataFolder + r'\Processed'
if not os.path.exists(processedFolder):
    os.makedirs(processedFolder)
df_all = pd.DataFrame(results)
print(df_all.head())
df_all.to_pickle(dataFolder + r'\Processed\FX40_Spectroscopy_Data_corrected.pkl')

print(f"Loaded {len(os.listdir(dataFolder))} ramp files")


C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\AFM\CAFM\-2V - 2V\Area 16/


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:\\Users\\newsonj\\OneDrive - Lancaster University\\Documents\\PhD\\Electrospray\\Fused Porphyrin\\fP2\\2026-10-01 new 100ugml 1hr\\AFM\\CAFM\\-2V - 2V\\Area 16/'

## Filter and calibrate files

In [14]:
calibrationFile = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-11-05 new 100ugml 1hr\AFM\CAFM\-2.5V - 3V\Calibration\Processed\FX40_Spectroscopy_Data.pkl"
calib = pd.read_pickle(calibrationFile)
currentAvg = np.nanmean(np.array(calib["Current"].tolist()),axis=0)

dataFolder = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-11-05 new 100ugml 1hr\AFM\CAFM\-2.5V - 3V\Area 2"

if dataFolder[-1] != "/":
    dataFolder = dataFolder + "/"

print(dataFolder)

SCBiasRange = [0,2] # define the bias range for filtering short circuits
OCBiasRange = [1.8,2] # define the bias range for filtering open circuits

maxCurrent = 9900 # max current in nA for filtering short circuits
minCurrent = 0.5 # min current in nA for filtering open circuits

results = []

files = sorted(
    f for f in os.listdir(dataFolder)
    if os.path.isfile(os.path.join(dataFolder, f)) and f.lower().endswith('.tiff')
)

reader = Read_tiff()

for file in tqdm(files):
    spectDict, spectTiff = reader(os.path.join(dataFolder,file))
    v = spectDict['SPECT_DATA'][0][0]
    j = spectDict['SPECT_DATA'][0][1]

    if any((SCBiasRange[0] < np.abs(v[i]) < SCBiasRange[1] and np.abs(j[i]) > maxCurrent) for i in range(len(v))):
        continue  # Skip this file if it meets the short circuit condition

    if any((OCBiasRange[0] < np.abs(v[i]) < OCBiasRange[1] and np.abs(j[i]) < minCurrent) for i in range(len(v))):
        continue  # Skip this file if it meets the open circuit condition

    j = j-currentAvg
    g = np.gradient(j*1e-9,v)
    logg = np.log10(np.abs(g/g_0))
    logj = np.log10(np.abs(j*1e-9))

    results.append({
        "filename": file,
        "Bias": v.tolist(),
        "Current": (j).tolist(), 
        "dIdV": g.tolist(),
        "Log(G/G0)": logg.tolist(),
        "Log(Current)": logj.tolist()
    })

processedFolder = dataFolder + r'\Processed'
if not os.path.exists(processedFolder):
    os.makedirs(processedFolder)
df_all = pd.DataFrame(results)
print(df_all.head())
df_all.to_pickle(dataFolder + r'\Processed\FX40_Spectroscopy_Data_Filtered_Calibrated.pkl')

print(f"Loaded {len(os.listdir(dataFolder))} ramp files")


C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-11-05 new 100ugml 1hr\AFM\CAFM\-2.5V - 3V\Area 2/


  0%|          | 0/9 [00:00<?, ?it/s]

Empty DataFrame
Columns: []
Index: []
Loaded 10 ramp files


# Memristor script loader

In [6]:
baseDir = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-11-05 new 100ugml 1hr\AFM\CAFM\Memristor measurements"

sequence = ['WIDE', 'WRITE', 'READW', 'ERASE', 'READE', 'WIDE']

calibrationFiles = [r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-11-05 new 100ugml 1hr\AFM\CAFM\Calibration\-1.5V - 2V\Processed\FX40_Spectroscopy_Data.pkl",#wide
                    r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-11-05 new 100ugml 1hr\AFM\CAFM\Calibration\-1.5V - 0V\Processed\FX40_Spectroscopy_Data.pkl",#write
                    r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-11-05 new 100ugml 1hr\AFM\CAFM\Calibration\-0.5V - 0V\Processed\FX40_Spectroscopy_Data.pkl",#read
                    r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-11-05 new 100ugml 1hr\AFM\CAFM\Calibration\-0.5V - 0V\Processed\FX40_Spectroscopy_Data.pkl",#read
                    r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-11-05 new 100ugml 1hr\AFM\CAFM\Calibration\0V - 2V\Processed\FX40_Spectroscopy_Data.pkl"]#erase

reader = Read_tiff()

calib = [pd.read_pickle(f) for f in calibrationFiles]
currentAvgs = {"WIDE": np.nanmean(np.array(calib[0]["Current"].tolist()),axis=0),
               "WRITE": np.nanmean(np.array(calib[1]["Current"].tolist()),axis=0),
               "READW": np.nanmean(np.array(calib[2]["Current"].tolist()),axis=0),
               "READE": np.nanmean(np.array(calib[3]["Current"].tolist()),axis=0),
               "ERASE": np.nanmean(np.array(calib[4]["Current"].tolist()),axis=0)}

folders = sorted(
    f for f in os.listdir(baseDir)
    if os.path.isdir(os.path.join(baseDir, f))
)

for folder in tqdm(folders):
    files = [f for f in os.listdir(os.path.join(baseDir, folder)) if f.endswith('.tiff')]
    files.sort(key=lambda f: int(os.path.splitext(f)[0].rstrip("_").rsplit("_", 1)[-1]))
    #print(files)

    filesSplit = [files[i:i+len(sequence)] for i in range(0, len(files), len(sequence))]
    print(folder + ": " + str(len(filesSplit)) + " loops")
    #print(filesSplit)

    data = []

    for group in filesSplit:
        valid = True
        groupData = []
        for i, file in enumerate(group):
            spectDict, spectTiff = reader(os.path.join(baseDir, folder, file))
            v = spectDict['SPECT_DATA'][0][0]
            j = spectDict['SPECT_DATA'][0][1]-currentAvgs[sequence[i]]
            g = np.gradient(j*1e-9,v)
            logg = np.log10(np.abs(g/g_0))
            logj = np.log10(np.abs(j*1e-9))
            if np.max(np.abs(j))>9900:
                valid = False
                print(f"Invalid data in {file}, skipping loop {group}")
                break
            groupData.append({
                "filename": file,
                "Type": sequence[i],
                "Bias": v.tolist(),
                "Current": (j).tolist(), 
                "dIdV": g.tolist(),
                "Log(G/G0)": logg.tolist(),
                "Log(Current)": logj.tolist()
            })
        if valid:
            data.extend(groupData)

    if not os.path.exists(os.path.join(baseDir, folder, 'Processed')):
        os.makedirs(os.path.join(baseDir, folder, 'Processed'))
    df_all = pd.DataFrame(data)
    df_all.to_pickle(os.path.join(baseDir, folder, 'Processed', 'FX40_Spectroscopy_Data_corrected.pkl'))




  0%|          | 0/424 [00:00<?, ?it/s]

0: 18 loops
Invalid data in iv_ 261106_108_.tiff, skipping loop ['iv_ 261106_103_.tiff', 'iv_ 261106_104_.tiff', 'iv_ 261106_105_.tiff', 'iv_ 261106_106_.tiff', 'iv_ 261106_107_.tiff', 'iv_ 261106_108_.tiff']
1: 3 loops
10: 1 loops
Invalid data in iv_ 261106_193_.tiff, skipping loop ['iv_ 261106_193_.tiff', 'iv_ 261106_194_.tiff', 'iv_ 261106_195_.tiff', 'iv_ 261106_196_.tiff', 'iv_ 261106_197_.tiff', 'iv_ 261106_198_.tiff']
100: 6 loops
101: 6 loops
102: 4 loops
Invalid data in iv_ 261106_1897_.tiff, skipping loop ['iv_ 261106_1897_.tiff', 'iv_ 261106_1898_.tiff', 'iv_ 261106_1899_.tiff', 'iv_ 261106_1900_.tiff', 'iv_ 261106_1901_.tiff', 'iv_ 261106_1902_.tiff']
103: 25 loops
Invalid data in iv_ 261106_1903_.tiff, skipping loop ['iv_ 261106_1903_.tiff', 'iv_ 261106_1904_.tiff', 'iv_ 261106_1905_.tiff', 'iv_ 261106_1906_.tiff', 'iv_ 261106_1907_.tiff', 'iv_ 261106_1908_.tiff']
Invalid data in iv_ 261106_1909_.tiff, skipping loop ['iv_ 261106_1909_.tiff', 'iv_ 261106_1910_.tiff', 'iv_ 2